# Sesion 12 — Solucion Guia: Analisis Actuarial de Autos
## Diplomado: Machine Learning en Seguros · FC UNAM
### Miercoles 13 de mayo de 2026  ·  18:00 - 21:00 h

---

> **Proposito de este notebook:** resolver en vivo el mismo ejercicio
> de 4 fases de S11, pero para el ramo **Autos**.
> Usalo como guia de como se espera que entregues tu analisis de GMM.

---

**Lo que vamos a resolver:**

| Fase | Herramienta | Pregunta |
|------|-------------|----------|
| 1 | NumPy | ¿Cuáles tipos de vehículo tienen mayor frecuencia y severidad? |
| 2 | SciPy | ¿Qué distribución ajusta mejor los siniestros de Autos? |
| 3 | NumPy Monte Carlo | ¿Es suficiente la prima cobrada para cubrir el P95 de siniestros? |
| 4 | Plotly | Dashboard ejecutivo con los hallazgos |

**¿Por que Autos es interesante?**

A diferencia de GMM, Autos tiene la variable `tipo_vehiculo` (Sedan, SUV, Hatchback, Pick-up).
Eso nos permite segmentar el analisis y detectar si la tarifa
deberia variar por tipo de vehiculo — algo real en cualquier aseguradora.

---
## Setup


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Carga el parquet que generaste en S9 — puede tener el nombre que le hayas dado
# Ajusta la ruta segun donde tengas guardado el archivo
import os
rutas_posibles = [
    '../sesion_10/datos/cartera_q1_2026_final.parquet',
    'datos/cartera_q1_2026_final.parquet',
]
ruta_ok = next((r for r in rutas_posibles if os.path.exists(r)), None)
if ruta_ok is None:
    raise FileNotFoundError(
        'No se encontro el archivo parquet.\n'
        'Ajusta la ruta en la variable rutas_posibles.'
    )
print(f'Cargando: {ruta_ok}')
df = pd.read_parquet(ruta_ok)

# Filtrar solo Autos
df_autos = df[df['ramo'] == 'Autos'].copy()
print(f'Polizas Autos: {len(df_autos):,}')
print(f'Columnas: {list(df_autos.columns)}')
print()
print('Tipos de vehiculo:')
print(df_autos['tipo_vehiculo'].value_counts())
print(f'NaN en tipo_vehiculo: {df_autos["tipo_vehiculo"].isna().sum()}')


Cargando: ../sesion_10/datos/cartera_q1_2026_final.parquet
Polizas Autos: 14,752
Columnas: ['id_poliza', 'num_poliza', 'nombre', 'apellido_paterno', 'apellido_materno', 'rfc', 'edad', 'sexo', 'estado_civil', 'ocupacion', 'ramo', 'plan', 'fecha_emision', 'fecha_inicio_vigencia', 'fecha_fin_vigencia', 'num_renovaciones', 'status_poliza', 'motivo_baja', 'canal_venta', 'marca_vehiculo', 'modelo_vehiculo', 'tipo_vehiculo', 'suma_asegurada', 'deducible', 'prima_neta', 'prima_total', 'forma_pago', 'agente_id', 'estado', 'municipio', 'codigo_postal', 'nombre_agente', 'region', 'tipo_agente', 'comision_pct', 'n_siniestros', 'monto_reclamado', 'monto_pagado', 'tiene_abierto', 'tipo_principal', 'prima_mensual', 'loss_ratio', 'comision_est', 'nivel_riesgo', 'anio_emision', 'mes_emision', 'trimestre', 'segmento_prima_fijo', 'cuartil_prima', 'cod_ramo', 'anio_poliza', 'ramo_desde_codigo']

Tipos de vehiculo:
tipo_vehiculo
Deportivo    1921
Lujo         1897
Compacto     1867
Pick-up      1846
SUV   

---
## Exploracion Inicial — Conocer los Datos Antes de Calcular

**Regla:** nunca calcules sin antes explorar. Detectar anomalias aqui
evita errores costosos en las fases siguientes.

In [2]:
# ── Perfil de la cartera Autos ───────────────────────────────────────────────
print('=== PERFIL CARTERA AUTOS ===')
print(f'Polizas:          {len(df_autos):,}')
print(f'Prima total:      ${df_autos["prima_total"].sum():,.0f}')
print(f'Prima promedio:   ${df_autos["prima_total"].mean():,.2f}')
print(f'Monto pagado:     ${df_autos["monto_pagado"].sum():,.0f}')
print(f'Loss ratio global:{df_autos["monto_pagado"].sum()/df_autos["prima_total"].sum():.4f}')
print()
print('NaN por columna clave:')
cols_clave = ['prima_total','monto_pagado','n_siniestros','tipo_vehiculo','edad','suma_asegurada']
for col in cols_clave:
    n = df_autos[col].isna().sum()
    if n > 0:
        print(f'  {col}: {n} NaN ({n/len(df_autos):.1%})')
    else:
        print(f'  {col}: sin NaN')

=== PERFIL CARTERA AUTOS ===
Polizas:          14,752
Prima total:      $253,823,506
Prima promedio:   $17,206.04
Monto pagado:     $64,499,502
Loss ratio global:0.2541

NaN por columna clave:
  prima_total: sin NaN
  monto_pagado: sin NaN
  n_siniestros: sin NaN
  tipo_vehiculo: sin NaN
  edad: sin NaN
  suma_asegurada: sin NaN


In [3]:
# ── Estadisticas por tipo de vehiculo — primera vista ─────────────────────────
resumen_vehiculo = df_autos.groupby('tipo_vehiculo').agg(
    polizas      = ('id_poliza',    'count'),
    prima_prom   = ('prima_total',  'mean'),
    prima_total  = ('prima_total',  'sum'),
    monto_pagado = ('monto_pagado', 'sum'),
    n_sins_total = ('n_siniestros', 'sum'),
).round(2).reset_index()

resumen_vehiculo['loss_ratio'] = (
    resumen_vehiculo['monto_pagado'] / resumen_vehiculo['prima_total']
).round(4)
resumen_vehiculo['pct_cartera'] = (
    resumen_vehiculo['prima_total'] / resumen_vehiculo['prima_total'].sum() * 100
).round(1)

print('Resumen por tipo de vehiculo:')
print(resumen_vehiculo.to_string(index=False))

# Observacion inicial: ¿hay diferencias en loss_ratio entre tipos?
# Si las hay, la tarifa deberia segmentarse por tipo de vehiculo.

Resumen por tipo de vehiculo:
tipo_vehiculo  polizas  prima_prom  prima_total  monto_pagado  n_sins_total  loss_ratio  pct_cartera
    Camioneta     1789    17159.37   30698106.6    7634040.40         646.0      0.2487         12.1
     Compacto     1867    17247.73   32201504.3    9055660.79         681.0      0.2812         12.7
    Deportivo     1921    17633.81   33874549.1    8410619.49         709.0      0.2483         13.3
    Hatchback     1771    17187.12   30438388.4    7508053.96         637.0      0.2467         12.0
         Lujo     1897    17178.73   32588056.9    7341571.46         646.0      0.2253         12.8
      Pick-up     1846    17289.23   31915923.9    8136284.35         674.0      0.2549         12.6
          SUV     1834    17155.14   31462523.4    8834572.29         635.0      0.2808         12.4
        Sedan     1827    16773.10   30644453.7    7578699.66         662.0      0.2473         12.1


---
## Fase 1 — Frecuencia y Severidad por Tipo de Vehiculo (NumPy)

**Pregunta:** ¿tienen todos los tipos de vehiculo el mismo riesgo?
Si el Pick-up tiene frecuencia 2x mayor que el Sedan,
deberian pagar una prima diferente.

**Prima pura = frecuencia × severidad**
- Frecuencia: ¿con qué probabilidad ocurre un siniestro?
- Severidad: si ocurre, ¿cuánto cuesta en promedio?

In [4]:
# ── Arrays NumPy para toda la cartera Autos ──────────────────────────────────
primas     = np.array(df_autos['prima_total'])
siniestros = np.array(df_autos['n_siniestros'])
montos     = np.array(df_autos['monto_pagado'])

# tipo_vehiculo puede ser Categorical — convertir a object antes del fillna
# para evitar el error 'Cannot setitem on a Categorical with a new category'
vehiculos = np.array(df_autos['tipo_vehiculo'].astype(object).fillna('Desconocido'))

# Tipos unicos — ignorar NaN
tipos_vehiculo = df_autos['tipo_vehiculo'].dropna().unique()
# Si es Categorical, extraer los valores como strings
if hasattr(tipos_vehiculo, 'categories'):
    tipos_vehiculo = tipos_vehiculo.astype(str)

print(f'Tipos de vehiculo a analizar: {list(tipos_vehiculo)}')
print(f'Total polizas Autos: {len(primas):,}')


Tipos de vehiculo a analizar: [np.str_('Compacto'), np.str_('Deportivo'), np.str_('Pick-up'), np.str_('Sedan'), np.str_('Camioneta'), np.str_('Lujo'), np.str_('Hatchback'), np.str_('SUV')]
Total polizas Autos: 14,752


In [5]:
# ── Calcular frecuencia y severidad por tipo de vehiculo ─────────────────────
resultados = []

for tipo in tipos_vehiculo:
    mask     = vehiculos == tipo
    p_tipo   = primas[mask]
    s_tipo   = siniestros[mask]
    m_tipo   = montos[mask]

    n_pol    = int(mask.sum())
    n_sin    = int(s_tipo.sum())

    # Frecuencia: siniestros totales / polizas del tipo
    frecuencia = n_sin / n_pol

    # Severidad: promedio de montos SOLO donde hubo pago
    # (incluir los ceros distorsionaria la severidad hacia abajo)
    pagos_positivos = m_tipo[m_tipo > 0]
    severidad = float(pagos_positivos.mean()) if len(pagos_positivos) > 0 else 0.0

    # Prima pura teorica
    prima_pura  = frecuencia * severidad

    # Prima real promedio que se esta cobrando
    prima_real  = float(p_tipo.mean())

    # Adecuacion: prima_real / prima_pura
    # > 1.0 = sobre-tarifado (se cobra mas de lo que el riesgo justifica)
    # < 1.0 = sub-tarifado  (se cobra menos — riesgo potencial para la aseguradora)
    adecuacion  = prima_real / prima_pura if prima_pura > 0 else None

    # Loss ratio: monto pagado total / prima cobrada total
    loss_ratio  = float(m_tipo.sum()) / float(p_tipo.sum()) if p_tipo.sum() > 0 else 0

    resultados.append(dict(
        tipo_vehiculo = tipo,
        polizas       = n_pol,
        siniestros    = n_sin,
        frecuencia    = round(frecuencia, 4),
        severidad     = round(severidad, 2),
        prima_pura    = round(prima_pura, 2),
        prima_real    = round(prima_real, 2),
        adecuacion    = round(adecuacion, 3) if adecuacion else None,
        loss_ratio    = round(loss_ratio, 4),
    ))

tabla_f1 = pd.DataFrame(resultados).sort_values('frecuencia', ascending=False)
print('=== TABLA FRECUENCIA-SEVERIDAD POR TIPO DE VEHICULO ===')
print(tabla_f1.to_string(index=False))

=== TABLA FRECUENCIA-SEVERIDAD POR TIPO DE VEHICULO ===
tipo_vehiculo  polizas  siniestros  frecuencia  severidad  prima_pura  prima_real  adecuacion  loss_ratio
    Deportivo     1921         709      0.3691   33375.47    12318.17    17633.81       1.432      0.2483
      Pick-up     1846         674      0.3651   36322.70    13261.92    17289.23       1.304      0.2549
     Compacto     1867         681      0.3648   36078.33    13159.80    17247.73       1.311      0.2812
        Sedan     1827         662      0.3623   33985.20    12314.29    16773.10       1.362      0.2473
    Camioneta     1789         646      0.3611   33929.07    12251.64    17159.37       1.401      0.2487
    Hatchback     1771         637      0.3597   34759.51    12502.43    17187.12       1.375      0.2467
          SUV     1834         635      0.3462   37593.92    13016.44    17155.14       1.318      0.2808
         Lujo     1897         646      0.3405   33370.78    11364.01    17178.73       1.512   

In [6]:
# ── Interpretacion de la Fase 1 ──────────────────────────────────────────────
# Esta celda muestra como escribir la interpretacion de los resultados
# En tu entrega de GMM debes tener una celda similar

vehiculo_max_frec = tabla_f1.loc[tabla_f1['frecuencia'].idxmax(), 'tipo_vehiculo']
vehiculo_max_sev  = tabla_f1.loc[tabla_f1['severidad'].idxmax(),  'tipo_vehiculo']
vehiculo_subtarif = tabla_f1.loc[tabla_f1['adecuacion'].idxmin(), 'tipo_vehiculo']
adec_min          = tabla_f1['adecuacion'].min()

print('=== HALLAZGOS DE LA FASE 1 ===')
print()
print(f'1. Mayor frecuencia de siniestros: {vehiculo_max_frec}')
print(f'   → Este tipo genera siniestros con mas frecuencia.')
print(f'   → Si la tarifa no lo refleja, la aseguradora subsidia a esos asegurados.')
print()
print(f'2. Mayor severidad promedio: {vehiculo_max_sev}')
print(f'   → Cuando tienen siniestro, el costo es mayor.')
print(f'   → Probablemente refleja el valor del vehiculo.')
print()
print(f'3. Mayor sub-tarifacion: {vehiculo_subtarif} (adecuacion={adec_min:.3f})')
print(f'   → Se cobra {adec_min:.1%} de lo que el riesgo justifica.')
print(f'   → Recomendacion: revisar la tarifa para este segmento.')

=== HALLAZGOS DE LA FASE 1 ===

1. Mayor frecuencia de siniestros: Deportivo
   → Este tipo genera siniestros con mas frecuencia.
   → Si la tarifa no lo refleja, la aseguradora subsidia a esos asegurados.

2. Mayor severidad promedio: SUV
   → Cuando tienen siniestro, el costo es mayor.
   → Probablemente refleja el valor del vehiculo.

3. Mayor sub-tarifacion: Pick-up (adecuacion=1.304)
   → Se cobra 130.4% de lo que el riesgo justifica.
   → Recomendacion: revisar la tarifa para este segmento.


In [7]:
# ── Visualizacion Fase 1: prima pura vs prima real por tipo ──────────────────
tabla_melt = tabla_f1.melt(
    id_vars='tipo_vehiculo',
    value_vars=['prima_pura', 'prima_real'],
    var_name='tipo', value_name='monto'
)

fig = px.bar(
    tabla_melt,
    x='tipo_vehiculo', y='monto', color='tipo',
    barmode='group',
    color_discrete_map={'prima_pura':'#F4A261', 'prima_real':'#1A5276'},
    title='Prima Pura Teorica vs Prima Real por Tipo de Vehiculo',
    labels={'monto':'Prima (MXN)', 'tipo_vehiculo':'Tipo de vehiculo'},
    text_auto='.0f',
)
fig.update_traces(textposition='outside')
fig.update_layout(height=430, uniformtext_minsize=9)
fig.show()

# Si la barra naranja (prima_pura) supera la azul (prima_real):
# el tipo esta sub-tarifado — se cobra menos de lo que el riesgo vale.

---
## Fase 2 — Distribucion de Siniestros de Autos (SciPy)

**Pregunta:** ¿qué distribución ajusta mejor los montos de siniestros de Autos?

Los siniestros de Autos típicamente incluyen colisiones, robo total y daños.
La distribución puede ser distinta a GMM porque los montos tienen
un techo natural (el valor del vehículo) a diferencia de los gastos médicos.

In [8]:
# ── Preparar los datos de siniestros de Autos ────────────────────────────────
montos_autos = df_autos.loc[df_autos['monto_pagado'] > 0, 'monto_pagado'].dropna().values

print(f'Siniestros con pago > 0: {len(montos_autos):,}')
print(f'Media:    ${montos_autos.mean():>12,.2f}')
print(f'Mediana:  ${np.median(montos_autos):>12,.2f}')
print(f'Max:      ${montos_autos.max():>12,.2f}')
print(f'Asimetria:{stats.skew(montos_autos):>12.3f}  (>0 = cola derecha)')
print()
print('Nota: la asimetria positiva justifica usar lognormal o gamma')
print('(distribuciones disenadas para variables positivas sesgadas)')

Siniestros con pago > 0: 1,846
Media:    $   34,940.14
Mediana:  $   20,516.06
Max:      $  888,331.81
Asimetria:       5.931  (>0 = cola derecha)

Nota: la asimetria positiva justifica usar lognormal o gamma
(distribuciones disenadas para variables positivas sesgadas)


In [9]:
# ── Ajustar 3 distribuciones y comparar por AIC ──────────────────────────────
# AIC = Akaike Information Criterion: penaliza modelos con mas parametros
# Menor AIC = mejor balance entre ajuste y parsimonia

distribuciones = {
    'Lognormal': stats.lognorm,
    'Gamma':     stats.gamma,
    'Pareto':    stats.pareto,
}

resultados_dist = []
params_guardados = {}

for nombre, dist in distribuciones.items():
    # fit() usa Maximum Likelihood Estimation
    # floc=0 fija el parametro de localizacion en 0
    params = dist.fit(montos_autos, floc=0)

    # Log-likelihood: que tan probable es observar estos datos con estos parametros
    ll = float(np.sum(dist.logpdf(montos_autos, *params)))

    # AIC = 2*k - 2*ll  donde k = numero de parametros
    aic = 2 * len(params) - 2 * ll

    # KS test: H0 = los datos siguen esta distribucion
    ks, pv = stats.kstest(montos_autos, dist.name, args=params)

    resultados_dist.append(dict(
        distribucion=nombre, aic=round(aic,1),
        ks_stat=round(ks,4), p_value=round(pv,4)
    ))
    params_guardados[nombre] = params

tabla_dist = pd.DataFrame(resultados_dist).sort_values('aic')
print('Comparacion de distribuciones (menor AIC = mejor ajuste):')
print(tabla_dist.to_string(index=False))
ganadora = tabla_dist.iloc[0]['distribucion']
print(f'\nDistribucion ganadora: {ganadora}')

Comparacion de distribuciones (menor AIC = mejor ajuste):
distribucion     aic  ks_stat  p_value
   Lognormal 42271.2   0.0474   0.0005
       Gamma 42307.2   0.0517   0.0001
      Pareto 46156.4   0.4014   0.0000

Distribucion ganadora: Lognormal


In [10]:
# ── Parametros de la distribucion ganadora ───────────────────────────────────
params_ln = params_guardados['Lognormal']
sigma_ln, loc_ln, escala_ln = params_ln
mu_ln = np.log(escala_ln)

media_teo  = stats.lognorm.mean(s=sigma_ln, loc=loc_ln, scale=escala_ln)
median_teo = stats.lognorm.median(s=sigma_ln, loc=loc_ln, scale=escala_ln)
p95_teo    = stats.lognorm.ppf(0.95, s=sigma_ln, loc=loc_ln, scale=escala_ln)

print('=== PARAMETROS LOGNORMAL AJUSTADA (AUTOS) ===')
print(f'mu    = {mu_ln:.4f}')
print(f'sigma = {sigma_ln:.4f}')
print()
print(f'{"":25} {"Observado":>12} {"Teorico":>12}')
print(f'{"Media":25} ${montos_autos.mean():>11,.0f} ${media_teo:>11,.0f}')
print(f'{"Mediana":25} ${np.median(montos_autos):>11,.0f} ${median_teo:>11,.0f}')
print(f'{"Percentil 95":25} ${np.percentile(montos_autos,95):>11,.0f} ${p95_teo:>11,.0f}')
print()
print('Prima pura teorica usando la lognormal:')

# La prima pura por poliza = frecuencia global × media de la lognormal
frec_global_autos = float(siniestros.sum()) / len(df_autos)
# Nota: aqui usamos siniestros de TODA la cartera Autos, no solo con pago
# Porque la frecuencia incluye los siniestros que no generaron pago
frec_autos = float(np.array(df_autos['n_siniestros']).sum()) / len(df_autos)

prima_pura_ln  = frec_autos * media_teo
prima_real_med = float(df_autos['prima_total'].mean())

print(f'  Frecuencia global Autos: {frec_autos:.4f}')
print(f'  Severidad (media lognorm): ${media_teo:,.2f}')
print(f'  Prima pura teorica:        ${prima_pura_ln:,.2f} por poliza')
print(f'  Prima real promedio:       ${prima_real_med:,.2f} por poliza')
print(f'  Adecuacion global:         {prima_real_med/prima_pura_ln:.3f}')

=== PARAMETROS LOGNORMAL AJUSTADA (AUTOS) ===
mu    = 9.8108
sigma = 1.2437

                             Observado      Teorico
Media                     $     34,940 $     39,503
Mediana                   $     20,516 $     18,229
Percentil 95              $    110,689 $    140,987

Prima pura teorica usando la lognormal:
  Frecuencia global Autos: 0.3586
  Severidad (media lognorm): $39,503.07
  Prima pura teorica:        $14,165.62 por poliza
  Prima real promedio:       $17,206.04 por poliza
  Adecuacion global:         1.215


In [11]:
# ── Visualizar ajuste: histograma + curva lognormal + Q-Q Plot ───────────────
x_range = np.linspace(
    montos_autos.min(),
    np.percentile(montos_autos, 99),
    300
)
pdf_ln = stats.lognorm.pdf(x_range, sigma_ln, loc_ln, escala_ln)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Datos vs Lognormal Ajustada',
        'Q-Q Plot — Ajuste de la Distribucion',
    ]
)

# Panel izquierdo: histograma + curva
fig.add_trace(go.Histogram(
    x=montos_autos, nbinsx=60,
    histnorm='probability density',
    marker_color='#1A5276', opacity=0.7, name='Datos Autos',
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=x_range, y=pdf_ln, mode='lines',
    line=dict(color='#F4A261', width=3),
    name='Lognormal ajustada',
), row=1, col=1)

fig.add_vline(
    x=media_teo, line_dash='dash', line_color='red',
    annotation_text=f'Media ${media_teo:,.0f}',
    row=1, col=1
)

# Panel derecho: Q-Q Plot
percentiles  = np.linspace(0.01, 0.99, 200)
cuant_teo    = stats.lognorm.ppf(percentiles, sigma_ln, loc_ln, escala_ln)
cuant_obs    = np.percentile(montos_autos, percentiles * 100)

fig.add_trace(go.Scatter(
    x=cuant_teo, y=cuant_obs, mode='markers',
    marker=dict(color='#1A5276', size=4, opacity=0.6),
    name='Cuantiles',
    hovertemplate='Teorico: $%{x:,.0f}<br>Observado: $%{y:,.0f}<extra></extra>',
), row=1, col=2)

lim = max(float(cuant_teo.max()), float(cuant_obs.max()))
fig.add_trace(go.Scatter(
    x=[0, lim], y=[0, lim], mode='lines',
    line=dict(color='red', dash='dash'), name='Ajuste perfecto',
), row=1, col=2)

fig.update_layout(
    height=430,
    title_text='Fase 2 — Ajuste Lognormal a Siniestros Autos',
)
fig.show()

# Como interpretar el Q-Q Plot:
# Los puntos siguen bien la diagonal hasta ~$200k
# En la cola extrema se separan — la lognormal subestima ligeramente los
# siniestros muy grandes (robo total de vehiculos de lujo).
# Para ese segmento especifico podria considerarse una Pareto.

---
## Fase 3 — Simulacion Monte Carlo (NumPy)

**Pregunta:** ¿la prima cobrada es suficiente para cubrir el escenario adverso?

Simulamos 10,000 anos posibles de la cartera Autos.
Cada ano puede tener mas o menos siniestros — queremos saber
en el peor 5% de los casos cuanto tendriamos que pagar (P95).

In [12]:
# ── Parametros de la simulacion ───────────────────────────────────────────────
np.random.seed(42)  # reproducibilidad — siempre fijarlo antes de simular

N_autos      = len(df_autos)
frec_autos   = float(np.array(df_autos['n_siniestros']).sum()) / N_autos
N_sim        = 10_000

print('=== PARAMETROS DE SIMULACION ===')
print(f'Polizas en cartera:  {N_autos:,}')
print(f'Frecuencia anual:    {frec_autos:.4f} ({frec_autos:.1%} de polizas tiene siniestro)')
print(f'mu lognormal:        {mu_ln:.4f}')
print(f'sigma lognormal:     {sigma_ln:.4f}')
print(f'Simulaciones:        {N_sim:,}')

=== PARAMETROS DE SIMULACION ===
Polizas en cartera:  14,752
Frecuencia anual:    0.3586 (35.9% de polizas tiene siniestro)
mu lognormal:        9.8108
sigma lognormal:     1.2437
Simulaciones:        10,000


In [13]:
# ── Simular 10,000 anos ───────────────────────────────────────────────────────
# Paso 1: para cada ano simulado, cuantas polizas tienen siniestro
# Binomial(N_polizas, probabilidad_siniestro)
n_sins_sim = np.random.binomial(n=N_autos, p=frec_autos, size=N_sim)

# Paso 2: para cada siniestro, simular el monto usando la lognormal ajustada
# (usamos list comprehension — para cada escenario simulamos n montos y los sumamos)
pagos_sim = np.array([
    np.random.lognormal(mu_ln, sigma_ln, n).sum() if n > 0 else 0.0
    for n in n_sins_sim
])

# Paso 3: calcular estadisticas de los 10,000 escenarios
prima_cobrada_total = float(df_autos['prima_total'].sum())

print('=== RESULTADOS DE LA SIMULACION ===')
for ptile in [50, 75, 90, 95, 99]:
    reserva = float(np.percentile(pagos_sim, ptile))
    sufic   = prima_cobrada_total / reserva
    alerta  = '' if sufic >= 1 else '  ← INSUFICIENTE'
    print(f'  Reserva P{ptile:>2}: ${reserva:>15,.0f}  |  Suficiencia: {sufic:.1%}{alerta}')

print()
print(f'Prima cobrada total: ${prima_cobrada_total:,.0f}')
reserva_p95 = float(np.percentile(pagos_sim, 95))
print(f'Reserva P95:         ${reserva_p95:,.0f}')
print(f'Suficiencia P95:     {prima_cobrada_total/reserva_p95:.1%}')

=== RESULTADOS DE LA SIMULACION ===
  Reserva P50: $    208,820,533  |  Suficiencia: 121.6%
  Reserva P75: $    213,018,898  |  Suficiencia: 119.2%
  Reserva P90: $    216,653,435  |  Suficiencia: 117.2%
  Reserva P95: $    219,016,583  |  Suficiencia: 115.9%
  Reserva P99: $    223,258,034  |  Suficiencia: 113.7%

Prima cobrada total: $253,823,506
Reserva P95:         $219,016,583
Suficiencia P95:     115.9%


In [14]:
# ── Visualizar la distribucion de pagos simulados ────────────────────────────
fig = px.histogram(
    x=pagos_sim / 1e6,
    nbins=80,
    title=f'Simulacion Monte Carlo — Cartera Autos ({N_sim:,} escenarios)',
    labels={'x': 'Pago total en el ano (M MXN)', 'y': 'Frecuencia'},
    color_discrete_sequence=['#1A5276'],
)

# Agregar lineas de percentiles
for ptile, col, pos in [
    (90,  'orange',  'top left'),
    (95,  'red',     'top right'),
    (99,  'darkred', 'top right'),
]:
    val = float(np.percentile(pagos_sim, ptile)) / 1e6
    fig.add_vline(
        x=val, line_dash='dash', line_color=col,
        annotation_text=f'P{ptile}={val:.1f}M',
        annotation_position=pos,
    )

# Linea de prima cobrada
fig.add_vline(
    x=prima_cobrada_total / 1e6,
    line_dash='solid', line_color='green', line_width=2.5,
    annotation_text=f'Prima cobrada={prima_cobrada_total/1e6:.1f}M',
    annotation_position='top left',
)

fig.update_layout(height=430)
fig.show()

---
## Fase 4 — Dashboard Ejecutivo (Plotly)

El dashboard resume los hallazgos de las 3 fases en una sola vista.
Es lo que le presentarias a la direccion de la aseguradora.

In [15]:
# ── Dashboard de 4 paneles ───────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Fase 1: Prima Pura vs Prima Real por Tipo de Vehiculo',
        'Fase 2: Distribucion Siniestros + Lognormal',
        'Fase 3: Monte Carlo — Distribucion de Pagos',
        'Loss Ratio por Tipo de Vehiculo',
    ]
)

# ── Panel 1,1: prima pura vs real por tipo ────────────────────────────────────
tabla_ord = tabla_f1.sort_values('prima_pura')
for col, color, name in [('prima_pura','#F4A261','Prima pura'),('prima_real','#1A5276','Prima real')]:
    fig.add_trace(go.Bar(
        x=tabla_ord['tipo_vehiculo'], y=tabla_ord[col],
        name=name, marker_color=color, showlegend=True,
    ), row=1, col=1)

# ── Panel 1,2: histograma + curva lognormal ───────────────────────────────────
fig.add_trace(go.Histogram(
    x=montos_autos, nbinsx=50, histnorm='probability density',
    marker_color='#1A5276', opacity=0.7, name='Siniestros', showlegend=True,
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=x_range, y=pdf_ln, mode='lines',
    line=dict(color='#F4A261', width=2.5), name='Lognormal', showlegend=True,
), row=1, col=2)

# ── Panel 2,1: Monte Carlo ────────────────────────────────────────────────────
fig.add_trace(go.Histogram(
    x=pagos_sim/1e6, nbinsx=80,
    marker_color='#1A5276', opacity=0.8, name='Escenarios', showlegend=False,
), row=2, col=1)
for ptile, col in [(95,'red'),(99,'darkred')]:
    fig.add_vline(x=np.percentile(pagos_sim,ptile)/1e6,
                  line_dash='dash', line_color=col,
                  annotation_text=f'P{ptile}', row=2, col=1)
fig.add_vline(x=prima_cobrada_total/1e6,
              line_color='green', line_width=2,
              annotation_text='Prima', row=2, col=1)

# ── Panel 2,2: loss ratio por tipo ───────────────────────────────────────────
lr_ord = tabla_f1.sort_values('loss_ratio')
colores_lr = ['#1E8449' if lr < 1 else '#C0392B' for lr in lr_ord['loss_ratio']]
fig.add_trace(go.Bar(
    x=lr_ord['loss_ratio'], y=lr_ord['tipo_vehiculo'],
    orientation='h', marker_color=colores_lr, showlegend=False,
    hovertemplate='%{y}: LR=%{x:.4f}<extra></extra>',
), row=2, col=2)
fig.add_vline(x=1.0, line_dash='dash', line_color='gray',
              annotation_text='Break-even', row=2, col=2)

fig.update_layout(
    height=700,
    title_text='Dashboard Actuarial — Ramo Autos Q1 2026',
    title_font_size=16,
    barmode='group',
)
fig.show()

---
## Resumen Ejecutivo — Como Debe Verse Tu Entrega

Despues de las graficas, siempre hay una celda de texto con los hallazgos.
Esto es lo que diferencia un analisis de un ejercicio.

In [16]:
# ── Generar el resumen ejecutivo automaticamente ──────────────────────────────
# En tu entrega de GMM, esta seccion debe estar llena con tus propios hallazgos

vehiculo_subtarif = tabla_f1.loc[tabla_f1['adecuacion'].idxmin()]
vehiculo_sobretarif= tabla_f1.loc[tabla_f1['adecuacion'].idxmax()]
vehiculo_mayor_lr  = tabla_f1.loc[tabla_f1['loss_ratio'].idxmax()]
suficiencia_p95    = prima_cobrada_total / reserva_p95

print('=' * 60)
print('  RESUMEN EJECUTIVO — RAMO AUTOS Q1 2026')
print('=' * 60)
print()
print('CARTERA')
print(f'  Polizas analizadas: {len(df_autos):,}')
print(f'  Prima total cobrada: ${prima_cobrada_total:,.0f}')
print(f'  Loss ratio global:   {df_autos["monto_pagado"].sum()/df_autos["prima_total"].sum():.4f}')
print()
print('TARIFACION (Fase 1)')
print(f'  Tipo mas sub-tarifado:   {vehiculo_subtarif["tipo_vehiculo"]}')
print(f'    Prima real / prima pura: {vehiculo_subtarif["adecuacion"]:.3f}')
print(f'    Recomendacion: revisar la tarifa al alza para este segmento')
print(f'  Tipo mas sobre-tarifado: {vehiculo_sobretarif["tipo_vehiculo"]}')
print(f'    Adecuacion: {vehiculo_sobretarif["adecuacion"]:.3f}')
print()
print('DISTRIBUCION DE SINIESTROS (Fase 2)')
print(f'  Distribucion ajustada: Lognormal (mejor AIC)')
print(f'  Prima pura teorica (lognormal): ${prima_pura_ln:,.2f} por poliza')
print(f'  Prima real promedio:            ${prima_real_med:,.2f} por poliza')
print(f'  Adecuacion global:              {prima_real_med/prima_pura_ln:.3f}')
print()
print('SUFICIENCIA DE RESERVAS (Fase 3)')
print(f'  Simulacion: {N_sim:,} escenarios de un ano')
print(f'  Reserva P95:     ${reserva_p95:,.0f}')
print(f'  Prima cobrada:   ${prima_cobrada_total:,.0f}')
etiqueta = 'SUFICIENTE' if suficiencia_p95 >= 1 else 'INSUFICIENTE'
print(f'  Suficiencia P95: {suficiencia_p95:.1%}  {etiqueta}')
print()
print('CONCLUSION')
if suficiencia_p95 >= 1:
    print('  La cartera Autos muestra suficiencia en escenario P95.')
else:
    print('  La cartera Autos NO es suficiente en escenario P95.')
    print('  Se recomienda revisar la tarifa o incrementar la reserva tecnica.')

  RESUMEN EJECUTIVO — RAMO AUTOS Q1 2026

CARTERA
  Polizas analizadas: 14,752
  Prima total cobrada: $253,823,506
  Loss ratio global:   0.2541

TARIFACION (Fase 1)
  Tipo mas sub-tarifado:   Pick-up
    Prima real / prima pura: 1.304
    Recomendacion: revisar la tarifa al alza para este segmento
  Tipo mas sobre-tarifado: Lujo
    Adecuacion: 1.512

DISTRIBUCION DE SINIESTROS (Fase 2)
  Distribucion ajustada: Lognormal (mejor AIC)
  Prima pura teorica (lognormal): $14,165.62 por poliza
  Prima real promedio:            $17,206.04 por poliza
  Adecuacion global:              1.215

SUFICIENCIA DE RESERVAS (Fase 3)
  Simulacion: 10,000 escenarios de un ano
  Reserva P95:     $219,016,583
  Prima cobrada:   $253,823,506
  Suficiencia P95: 115.9%  SUFICIENTE

CONCLUSION
  La cartera Autos muestra suficiencia en escenario P95.


---
## Recursos y Siguientes Pasos

---

### Para Empezar Desde Cero

Si sientes que necesitas repasar o ir más despacio, estos recursos
son gratuitos, interactivos y no requieren instalar nada:

- **W3Schools Python** — https://www.w3schools.com/python/
  El tutorial más accesible que existe. Cada concepto tiene un editor
  "Try it Yourself" donde corres el código en el navegador sin instalar nada.
  Empieza aquí si algo del módulo quedó poco claro.

- **Kaggle — Python** — https://www.kaggle.com/learn/python
  Curso gratuito de Python desde cero, con ejercicios en el navegador.
  Sin registro previo puedes ver las lecciones.

- **Kaggle — Pandas** — https://www.kaggle.com/learn/pandas
  El curso de pandas de Kaggle. Cubre exactamente lo del T5 de este módulo
  con ejercicios resueltos inmediatamente en el navegador.

- **Kaggle — Data Visualization** — https://www.kaggle.com/learn/data-visualization
  Matplotlib y Seaborn desde cero con ejercicios. Complementa lo del T6.
  
### Libros

**Para empezar — directamente con lo que ya saben:**

- **Statistical Foundations of Actuarial Learning** — Wüthrich & Merz (Springer, 2023)
  Open access gratuito: https://link.springer.com/book/10.1007/978-3-031-12409-9
  El libro de referencia de ML aplicado a seguros. GLM, distribuciones y pricing.
  Los capítulos 2-4 son continuación directa de lo que hicimos en S11-S12.

- **CAS Open Source Python Tools** — Casualty Actuarial Society
  GitHub: https://github.com/casact
  Repositorios con código Python actuarial: chainladder (reservas), ActSim
  (simulación de siniestros), y painless_glms (GLM para pricing).

- **Actuarial Data Science** — Wüthrich & Merz
  Capitulos gratuitos: [actuarialdatascience.org](https://www.actuarialdatascience.org)
  El mas completo en modelos actuariales con Python. Capitulo 2 cubre exactamente
  frecuencia-severidad con distribuciones.

**Para profundizar en distribuciones y reservas:**

- **Loss Models: From Data to Decisions** — Klugman, Panjer & Willmot (Wiley)
  El clasico de actuaria de danos. Los capitulos 4-8 son lo que hicimos en S11-S12
  pero con toda la teoria detras.

**Para el Modulo 2 en adelante:**

- **An Introduction to Statistical Learning (ISLR)** — James et al.
  Gratuito en PDF y con labs en Python: [statlearning.com](https://www.statlearning.com)
  Capitulos 3 (regresion), 4 (clasificacion) y 8 (arboles) son los mas relevantes.

- **Hands-On Machine Learning** — Aurelien Geron (O'Reilly)
  El mejor libro practico de ML con sklearn. Codigo en GitHub:
  [github.com/ageron/handson-ml3](https://github.com/ageron/handson-ml3)

---
### Recursos Interactivos

**Cursos con ejercicios en el navegador — no requieren instalar nada:**

- **Kaggle Learn** — [kaggle.com/learn](https://www.kaggle.com/learn)
  Cursos gratuitos de pandas, visualizacion, ML e intro a deep learning.
  Cada leccion tiene ejercicios que corres directamente en el navegador.
  Recomendados: *Pandas*, *Data Visualization*, *Intro to Machine Learning*.

- **Google Colab** — [colab.research.google.com](https://colab.research.google.com)
  Jupyter Notebooks en la nube, gratis, con GPU. Util para practicar sin
  configurar nada. Guarda tus notebooks en Google Drive.

- **scipy.stats interactivo** — [docs.scipy.org/doc/scipy/reference/stats.html](https://docs.scipy.org/doc/scipy/reference/stats.html)
  Cada distribucion tiene ejemplos ejecutables. Util para explorar parametros.

- **Plotly ejemplos** — [plotly.com/python](https://plotly.com/python)
  Cada tipo de grafica tiene codigo copiar-pegar listo. Busca 'choropleth' o
  'violin' y tienes el codigo completo en segundos.

- **sklearn user guide** — [scikit-learn.org/stable/user_guide.html](https://scikit-learn.org/stable/user_guide.html)
  Cada modelo tiene ejemplos interactivos. Cuando lleguen a GLM en Modulo 2,
  busquen `sklearn.linear_model.PoissonRegressor`.

---
### Videos

- **StatQuest with Josh Starmer** — [youtube.com/@statquest](https://www.youtube.com/@statquest)
  El mejor canal para entender estadistica y ML desde intuicion.
  Videos clave para este diplomado:
  *Linear Regression*, *Logistic Regression*, *Decision Trees*, *Random Forests*.

- **3Blue1Brown** — [youtube.com/@3blue1brown](https://www.youtube.com/@3blue1brown)
  Matematicas visuales. La serie *Essence of Linear Algebra* es fundamental
  para entender por que los modelos funcionan.

- **Actuaries Institute YouTube** — [youtube.com/@ActuariesInstitute](https://www.youtube.com/channel/UCWj3gsgnMMEFqRb4SD3854Q)
  Presentaciones de conferencias actuariales. Muchas con Python y ML aplicado.
  Busca 'machine learning pricing' para casos reales de aseguradoras.

---
### Que Tipo de Ejercicios Conviene Resolver

**Para consolidar lo del Modulo 1:**

Toma tu analisis de GMM (la tarea) y extiendelo:
- Segmenta por `g_edad` en lugar de ramo — ¿cambia la adecuacion por grupo de edad?
- Cambia la distribucion — prueba Gamma en lugar de Lognormal y compara el AIC
- Modifica el Monte Carlo — ¿que pasa con la suficiencia si la frecuencia sube 20%?
  Eso simula un escenario adverso (pandemia, crisis economica)

**Para prepararse para el Modulo 2:**

Estos ejercicios de Kaggle son directamente relevantes:
- [kaggle.com/competitions/porto-seguro-safe-driver-prediction](https://www.kaggle.com/competitions/porto-seguro-safe-driver-prediction)
  Prediccion de siniestros de Autos — exactamente lo que hicimos pero a escala real
- [kaggle.com/competitions/homesite-quote-conversion](https://www.kaggle.com/competitions/homesite-quote-conversion)
  Conversion de cotizaciones de seguros de hogar

**Para ir mas alla:**

- Implementa un GLM de Poisson para modelar la frecuencia de siniestros
  (esto es exactamente lo que hace el Modulo 2)
- Carga datos reales de la CNSF (Comision Nacional de Seguros y Fianzas):
  [gob.mx/cnsf](https://www.gob.mx/cnsf) — tienen estadisticas publicas de la industria

---
### Que Sigue — Modulo 2

La base que construyeron aqui es exactamente lo que necesitan:

| Modulo 2 | Conexion con Modulo 1 |
|----------|----------------------|
| Analisis exploratorio avanzado | pandas + visualizacion |
| GLM de frecuencia y severidad | sklearn + SciPy |
| Feature engineering actuarial | NumPy + pandas |
| Dashboards interactivos | Plotly |

---
*Diplomado ML en Seguros · Facultad de Ciencias, UNAM · Modulo 1 completado · 2026*
